In [ ]:
import numpy as np
import math
import random
from typing import List, Dict, Any, Tuple
import sys
import os
from random import Random
import pandas as pd

sys.path.append(os.path.join(os.getcwd(), "TSP_CYTHON"))

from two_opt_cand import two_opt_cand as two_opt_cy
from three_opt_cand import three_opt_cy
from knn_cand import build_candidate_lists_knn as cython_knn

# ============================
# GLOBAL CONFIG
# ============================
PROBLEM_TABLE = "Problem Specific Values.csv"

# a280-n279
# a280-n1395
# a280-n2790
# fnl4461-n4460
# fnl4461-n22300
# fnl4461-n44600
# pla33810-n33809
# pla33810-n169045
# pla33810-n338090
PROBLEM_INSTANCE = "a280-n279"
INSTANCE_PATH = os.path.join(os.getcwd(), "gecco19-thief", "src", "main", "resources", PROBLEM_INSTANCE+".txt")

def load_problem_settings(instance_path: str):
    """
    Loads configuration values for a given TTP instance using an external CSV table. 

    Args:
        instance_path (str): Path to the TTP instance file.

    Returns:
        tuple: A tuple containing:
            - max_archive (int): Maximum archive size for the instance.
            - ideal (tuple): (ideal_time, ideal_profit) reference point.
            - nadir (tuple): (nadir_time, nadir_profit) reference point.
            - p_min (float): Minimum profit value for graph scaling.
            - p_max (float): Maximum profit value for graph scaling.
            - fname (str): Filename of the instance.
    """
    fname = os.path.basename(instance_path)

    df = pd.read_csv(PROBLEM_TABLE)

    # Strip spaces for safety
    df['FILE'] = df['FILE'].astype(str).str.strip()

    if fname not in df['FILE'].values:
        raise ValueError(f"Instance '{fname}' not found in problem-specific CSV")

    row = df[df['FILE'] == fname].iloc[0]

    max_archive = int(row['MAX_ARCHIVE'])

    ideal = (float(row['IDEAL_TIME']), float(row['IDEAL_PROFIT']))
    nadir = (float(row['NADIR_TIME']), float(row['NADIR_PROFIT']))

    # Profit graph values
    p_min = 0.0
    p_max = float(row['IDEAL_PROFIT_GRAPH'])

    return max_archive, ideal, nadir, p_min, p_max, fname

def load_parameters(instance: str):
    """
    Loads algorithm parameters for a given TTP instance from a CSV file.
    
    Args:
        instance (str): The name of the TTP instance.
        
    Returns:
        tuple: A tuple containing all the parameters in order.
    """
    df = pd.read_csv("parameters.csv")
    df['INSTANCE'] = df['INSTANCE'].astype(str).str.strip()
    if instance not in df['INSTANCE'].values:
        raise ValueError(f"Instance '{instance}' not found in parameters CSV")
    row = df[df['INSTANCE'] == instance].iloc[0]
    return tuple(int(row[col]) for col in df.columns if col != 'INSTANCE')

# Load problem-specific settings
MAX_ARCHIVE, IDEAL, NADIR, P_min, P_max, File = load_problem_settings(INSTANCE_PATH)
(
    # TSP / 2-OPT
    CAND_K,
    TWO_OPT_PASSES,
    # EVOLUTION
    POP_SIZE,
    GENERATIONS,
    TOURN_K,
    # ELITES
    TAIL_POP,
    SPARSE_POP,
    SPARSE_ELITES,
    TAIL_ELITES,
    MID_ELITES,
    # LKH-style TSP (multi-start + 3-opt + ILS)
    NUM_STARTS_LKH,
    ILS_RESTARTS_LKH,
    TWO_OPT_MAX_ITERS,
    THREE_OPT_TRIES_PER_LS,
    DB_CUTS,
    ELITE_2OPT_STEPS,
    ELITE_3OPT_STEPS,
    ELITE_ILS_PASSES,
    ELITE_DB_CUTS,
    HC_STEPS,
    NUM_GREEDIES,
) = load_parameters(PROBLEM_INSTANCE)

# ILS KICK / LS GLOBALS 
ILS_KICK_CUTS_BASE = DB_CUTS
ILS_KICK_CUTS_MID  = DB_CUTS + 1
ILS_KICK_CUTS_LATE = DB_CUTS + 2

ILS_LS_2OPT_EARLY = TWO_OPT_MAX_ITERS
ILS_LS_2OPT_MID   = TWO_OPT_MAX_ITERS // 2
ILS_LS_2OPT_LATE  = TWO_OPT_MAX_ITERS // 3

ILS_LS_3OPT_EARLY = THREE_OPT_TRIES_PER_LS
ILS_LS_3OPT_MID   = THREE_OPT_TRIES_PER_LS // 2
ILS_LS_3OPT_LATE  = THREE_OPT_TRIES_PER_LS // 3


VERBOSE = True

# Item seeding parameters
CAP_FACTORS = [0.03, 0.05, 0.075, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.0]
GAMMAS      = [0, 0.25, 0.5, 0.75, 1, 1.5, 2, 3, 4, 6, 8, 10, 15, 20, 30]
HEUR_MODES = ["suffix", "profit_adj", "time_ratio"]
MUT_RATE = 0.003

# Random seeds
GLOBAL_SEED = 0
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
rng_global      = Random(GLOBAL_SEED)
rng_crossover   = Random(GLOBAL_SEED + 1)
rng_mutation    = Random(GLOBAL_SEED + 2)
rng_tourkick    = Random(GLOBAL_SEED + 3)
rng_hillclimb   = Random(GLOBAL_SEED + 4)
rng_selection   = Random(GLOBAL_SEED + 5)
rng_greedyseed  = Random(GLOBAL_SEED + 6)

# Archive globals
archive_pop = [] # stores (tour, packing_plan)
archive_objs = [] # stores (T,P,W)

# Cache: tour tuple -> (dists_array, tsp_int_length)
TOUR_DIST_CACHE = {}

# Cache: (tour_key, chosen_tuple) -> (T, P, W)
EVAL_CACHE = {}

MAX_TOUR_CACHE = 2000

# Global elites across all generations
bestP = -1.0
bestP_sol = None

bestT = None
bestT_sol = None

In [ ]:
# ===============================================================
# RESET STATE BETWEEN RUNS
# ===============================================================

def reset_ttp_state():
    """
    Resets all global state, including caches, archives, and elite solutions. 
    """
    global archive_pop, archive_objs
    global bestP, bestP_sol, bestT, bestT_sol
    global TOUR_DIST_CACHE, EVAL_CACHE
    global run_evo

    # Reset caches
    TOUR_DIST_CACHE = {}
    EVAL_CACHE = {}

    # Reset archives
    archive_pop = []
    archive_objs = []

    # Reset elites
    bestP = -1.0
    bestP_sol = None
    bestT = None
    bestT_sol = None

    # Reset tour cache
    if hasattr(run_evo, "_tour_cache"):
        run_evo._tour_cache = {}

    print("[STATE] Caches and archives reset. TSP tour remains intact.")

In [ ]:

# ===============================================================
# PARSE INSTANCE
# ===============================================================
def parse_ttp(path: str) -> Dict[str, Any]:
    """
    Parses a TTP instance file and extracts all information, including city coordinates, item definitions, knapsack constraints,
    and speed parameters.

    Args:
        path (str): Path to the TTP instance file.

    Returns:
        Dict[str, Any]: A dictionary containing:
            - dim (int): Number of cities.
            - coords (ndarray): Array of shape (dim, 2) with city coordinates.
            - items (list): List of tuples (profit, weight, city_index).
            - capacity (float): Knapsack capacity.
            - v_min (float): Minimum travel speed.
            - v_max (float): Maximum travel speed.
            - edge_type (str): Type of edge weight metric used.
    """
    with open(path, "r") as f:
        lines = [line.strip() for line in f if line.strip()]

    dim = num_items = None
    capacity = v_min = v_max = None
    edge_type = None

    i = 0
    while i < len(lines):
        L = lines[i]
        if L.startswith("DIMENSION"):
            dim = int(L.split(":")[1])
        elif L.startswith("NUMBER OF ITEMS"):
            num_items = int(L.split(":")[1])
        elif L.startswith("CAPACITY OF KNAPSACK"):
            capacity = float(L.split(":")[1])
        elif L.startswith("MIN SPEED"):
            v_min = float(L.split(":")[1])
        elif L.startswith("MAX SPEED"):
            v_max = float(L.split(":")[1])
        elif L.startswith("EDGE_WEIGHT_TYPE"):
            edge_type = L.split(":")[1].strip()
        elif L.startswith("NODE_COORD_SECTION"):
            break
        i += 1

    i += 1
    coords = np.zeros((dim, 2), float)
    for k in range(dim):
        idx, x, y = lines[i + k].split()
        coords[int(idx) - 1] = [float(x), float(y)]
    i += dim

    while "ITEMS SECTION" not in lines[i]:
        i += 1
    i += 1

    items = []
    for j in range(num_items):
        parts = lines[i + j].split()
        profit = float(parts[1])
        weight = float(parts[2])
        city = int(parts[3]) - 1
        items.append((profit, weight, city))

    return {
        "dim": dim,
        "coords": coords,
        "items": items,
        "capacity": capacity,
        "v_min": v_min,
        "v_max": v_max,
        "edge_type": edge_type,
    }

# ===============================================================
# TSP HELPERS (distance matrix, 3-opt, ILS)
# ===============================================================

def compute_dists(order: np.ndarray, coords: np.ndarray) -> np.ndarray:
    """
    Computes the pairwise distances along a cyclic tour defined by a permutation
    of city indices.

    Args:
        order (np.ndarray): Array of city indices specifying the tour order.
        coords (np.ndarray): Array of shape (n, 2) containing city coordinates.

    Returns:
        np.ndarray: Array of length n where each element is the distance from city i to its successor in the tour.
    """
    n = len(order)
    out = np.empty(n, float)
    for i in range(n):
        a = order[i]
        b = order[(i + 1) % n]
        d = math.sqrt(((coords[a] - coords[b]) ** 2).sum())
        out[i] = math.ceil(d)
    return out

def build_candidate_lists_knn(coords: np.ndarray, k: int):
    """
    Builds k-nearest-neighbor candidate lists for each city using Cython-based KNN implementation.

    Args:
        coords (np.ndarray): Array of shape (n, 2) containing city coordinates.
        k (int): Number of nearest neighbors to return for each city.

    Returns:
        list: A list of k integers representing the indices of the nearest neighbors for that city.
    """

    cand = cython_knn(coords, k)
    return cand

def tour_length_list(tour, coords, accurate: bool = False) -> float:
    """
    Calculates the total length of a tour based on city coordinates.

    Args:
        tour (list or np.ndarray): Sequence of city indices representing the tour.
        coords (np.ndarray): Array of city coordinates (n x 2).
        accurate (bool): If False, uses fast integer distance (for LKH search). 
                         If True, computes real Euclidean distance. Defaults to False.

    Returns:
        float: Total length of the tour.
    """
    total = 0.0
    for i in range(len(tour)):
        a = tour[i]
        b = tour[(i + 1) % len(tour)]
        dx = coords[a][0] - coords[b][0]
        dy = coords[a][1] - coords[b][1]

        dist = math.hypot(dx, dy)

        if accurate:
            total += dist
        else:
            total += int(dist) 

    return total

def get_tour_dists(tour, coords):
    """
    Computes the distances between consecutive cities in a tour and caches the result.

    Args:
        tour (list or np.ndarray): Sequence of city indices representing the tour.
        coords (np.ndarray): Array of city coordinates (n x 2).

    Returns:
        tuple: 
            - np.ndarray: Array of distances between consecutive cities.
            - float: Total tour length (sum of distances).
    """
    key = tuple(tour)
    if key in TOUR_DIST_CACHE:
        return TOUR_DIST_CACHE[key]

    # compute once
    d = compute_dists(tour, coords)
    tsp_len = float(d.sum())

    TOUR_DIST_CACHE[key] = (d, tsp_len)
    return d, tsp_len

def nn_tour(coords: np.ndarray) -> np.ndarray:
    """
    Constructs a tour using the nearest-neighbor.

    Args:
        coords (np.ndarray): Array of city coordinates (n x 2).

    Returns:
        np.ndarray: Array of city indices representing the tour.
    """
    n = len(coords)
    unvis = np.ones(n, dtype=bool)
    tour = np.empty(n, int)
    cur = 0

    for t in range(n):
        tour[t] = cur
        unvis[cur] = False
        if t == n - 1:
            break
        dx = coords[unvis, 0] - coords[cur, 0]
        dy = coords[unvis, 1] - coords[cur, 1]
        nxt = np.where(unvis)[0][np.argmin(dx * dx + dy * dy)]
        cur = nxt

    return tour

def ensure_valid_tour(tour: List[int], n: int) -> List[int]:
    """
    Repairs a tour to ensure it is a valid permutation of city indices.

    The returned tour will:
    - Contain each city index exactly once.
    - Start at city index 0.

    Args:
        tour (List[int]): List of city indices representing a possibly invalid tour.
        n (int): Total number of cities.

    Returns:
        List[int]: Validated and repaired tour starting at city 0.
    """
    seen = set()
    cleaned = []
    for c in tour:
        if 0 <= c < n and c not in seen:
            cleaned.append(c)
            seen.add(c)
    for c in range(n):
        if c not in seen:
            cleaned.append(c)
            seen.add(c)
    if 0 in cleaned:
        k = cleaned.index(0)
        cleaned = cleaned[k:] + cleaned[:k]
    else:
        cleaned = [0] + [c for c in range(1, n)]
    return cleaned[:n]

def double_bridge_move(tour: List[int], cuts: int = DB_CUTS) -> List[int]:
    """
    Performs a double-bridge perturbation on a tour for diversification.

    The tour is split into segments at random cut points, the interior segments 
    are shuffled (and optionally reversed), and then the tour is reassembled.

    Args:
        tour (List[int]): Current tour as a list of city indices.
        cuts (int, optional): Number of cut points to use for the double-bridge move. 
                              Defaults to DB_CUTS.

    Returns:
        List[int]: New tour after applying the double-bridge move.
    """
    global rng_tourkick

    n = len(tour)
    if cuts < 2 or n < cuts + 2:
        return tour[:]  # nothing to do

    pts = sorted(rng_tourkick.sample(range(1, n), cuts))

    # split into segments
    segs = []
    prev = 0
    for p in pts:
        segs.append(tour[prev:p])
        prev = p
    # last segment
    segs.append(tour[prev:])  # last segment

    # nothing to permute
    if len(segs) <= 2:
        return tour[:] 

    # interior segments only
    core = segs[1:-1] 
    rng_tourkick.shuffle(core)

    for i in range(len(core)):
        if rng_tourkick.random() < 0.3: 
            core[i] = core[i][::-1]

    return segs[0] + sum(core, []) + segs[-1]

def build_candidate_lists_coordwise(coords: np.ndarray, k: int):
    """
    Builds memory-efficient k-nearest-neighbor candidate lists for TSP using coordinates.
    Avoids storing the full n x n distance matrix by computing squared Euclidean distances on the fly.

    Args:
        coords (np.ndarray): Array of city coordinates (n x 2).
        k (int): Number of nearest neighbors per city.

    Returns:
        list[list[int]]: List of n lists, each containing k nearest neighbor city indices.
    """
    n = len(coords)
    k = min(k, n - 1)

    cand = []
    for i in range(n):
        dx = coords[:,0] - coords[i,0]
        dy = coords[:,1] - coords[i,1]
        dist2 = dx*dx + dy*dy

        # exclude self
        dist2[i] = np.inf

        # find k smallest
        nn = np.argpartition(dist2, k)[:k]
        
        # sort them for stability
        nn = nn[np.argsort(dist2[nn])]
        cand.append(nn.tolist())

    return cand

def lkh_style_tsp(inst: Dict[str, Any], initial_tour: List[int]) -> Tuple[List[int], float, list]:
    """
    Improves a TSP tour using a KNN-based LKH-style iterative local search.

    The method interleaves 2-opt and 3-opt moves, uses double-bridge perturbations
    for diversification, and candidate lists for efficiency. Multiple randomised
    starts and ILS restarts are performed to escape local optima.

    Args:
        inst (Dict[str, Any]): TSP instance dictionary containing at least:
            - 'coords' (np.ndarray): City coordinates (n x 2).
        initial_tour (List[int]): Initial tour as a list of city indices.

    Returns:
        Tuple[List[int], float, list]:
            - List[int]: Best tour found after LKH-style optimization.
            - float: Length of the best tour.
            - list[list[int]]: Candidate lists used during optimization.
    """

    coords = inst["coords"]
    n = len(coords)

    # Build TRUE candidate lists using TSP metric 
    cand = build_candidate_lists_coordwise(coords, CAND_K)
    print(f"[CAND] Built coordwise KNN candidate lists (k={CAND_K})")

    best_global_tour = None
    best_global_len = float("inf")

    for s in range(NUM_STARTS_LKH):
        print(f"\n[START {s+1}/{NUM_STARTS_LKH}] using supplied NN tour...")
        base_tour = ensure_valid_tour(initial_tour[:], n)
        _, base_len = get_tour_dists(base_tour, coords)
        print(f"[BASE {s+1}] initial = {base_len}")

        # LOCAL SEARCH (INTERLEAVED 2-OPT -> 3-OPT -> 2-OPT)
        improved = True
        tour_ls = base_tour[:]

        while improved:
            improved = False

            # Pass 1: strong 2-opt
            tour_ls, _ = two_opt_cy(
                tour_ls, cand, coords,
                max_iter=TWO_OPT_MAX_ITERS,
                verbose=False,
                seed=rng_global.randint(0, 2**31 - 1)
            )

            # Pass 2: strong 3-opt
            new_ls, _ = three_opt_cy(
                tour_ls, cand, coords,
                max_iter=THREE_OPT_TRIES_PER_LS,
                verbose=False,
                seed=rng_global.randint(0, 2**31 - 1)
            )

            if new_ls != tour_ls:
                improved = True
                tour_ls = new_ls

            # Pass 3: final 2-opt cleanup
            tour_ls, _ = two_opt_cy(
                tour_ls, cand, coords,
                max_iter=TWO_OPT_MAX_ITERS // 2,   # cleanup budget
                verbose=False,
                seed=rng_global.randint(0, 2**31 - 1)
            )

        _, len_ls = get_tour_dists(tour_ls, coords)

        print(f"[BASIN {s+1}] {len_ls}")

        best_basin_tour = tour_ls[:]
        best_basin_len  = len_ls

        # FULL ILS LOOP (GLOBAL-PARAM VERSION)
        for r in range(ILS_RESTARTS_LKH):
            print(f"[ILS S{s+1}] restart {r+1}/{ILS_RESTARTS_LKH}")

            # Select kick strength by phase
            if r < ILS_RESTARTS_LKH // 3:
                cuts = max(4, ILS_KICK_CUTS_BASE)
            elif r < 2 * ILS_RESTARTS_LKH // 3:
                cuts = max(4, ILS_KICK_CUTS_MID)
            else:
                cuts = max(4, ILS_KICK_CUTS_LATE)

            kicked = ensure_valid_tour( double_bridge_move(best_basin_tour, cuts=cuts),n)
            tour_k = kicked[:]

            # LS strength also phased
            if r < ILS_RESTARTS_LKH // 3:
                max2 = ILS_LS_2OPT_EARLY
                max3 = ILS_LS_3OPT_EARLY
            elif r < 2 * ILS_RESTARTS_LKH // 3:
                max2 = ILS_LS_2OPT_MID
                max3 = ILS_LS_3OPT_MID
            else:
                max2 = ILS_LS_2OPT_LATE
                max3 = ILS_LS_3OPT_LATE

            # Pass 1: 2-opt
            tour_k, _ = two_opt_cy(
                tour_k, cand, coords,
                max_iter=max2,
                verbose=False,
                seed=rng_tourkick.randint(0, 2**31 - 1)
            )

            # Pass 2: 3-opt
            tour_k, _ = three_opt_cy(
                tour_k, cand, coords,
                max_iter=max3,
                verbose=False,
                seed=rng_tourkick.randint(0, 2**31 - 1)
            )

            # Pass 3: cleanup 2-opt
            tour_k, _ = two_opt_cy(
                tour_k, cand, coords,
                max_iter=ILS_LS_2OPT_LATE,
                verbose=False,
                seed=rng_tourkick.randint(0, 2**31 - 1)
            )

            # Evaluate
            _, k_len = get_tour_dists(tour_k, coords)

            if k_len < best_basin_len:
                print(f"[ILS S{s+1}] improved {best_basin_len} -> {k_len}")
                best_basin_len  = k_len
                best_basin_tour = tour_k[:]
            else:
                print(f"[ILS S{s+1}] no improvement ({k_len} >= {best_basin_len})")

        if best_basin_len < best_global_len:
            print(f"[GLOBAL] improved {best_global_len} -> {best_basin_len}")
            best_global_len  = best_basin_len
            best_global_tour = best_basin_tour[:]

    # FINAL POLISH
    best_global_tour, _ = two_opt_cy(
        best_global_tour,
        cand,
        coords,
        max_iter=TWO_OPT_PASSES,
        verbose=False,
        seed=rng_global.randint(0, 2**31 - 1)
    )

    _, best_global_len = get_tour_dists(best_global_tour, coords)

    print(f"\n[FINAL TSP] {best_global_len}")
    return best_global_tour, best_global_len, cand


# ===============================================================
# PRECOMPUTE ITEM DATA
# ===============================================================
def prep_items(inst, order, dists):
    """
    Prepares item data for the TTP along a given tour.

    Args:
        inst (dict): TTP instance dictionary.
        order (np.ndarray): Tour order as a sequence of city indices.
        dists (np.ndarray): Distances between consecutive cities in the tour.

    Returns:
        tuple:
            - list[dict]: List of item records, each containing:
                - 'index' (int): Item index.
                - 'city' (int): City index.
                - 'pos' (int): Position of the city in the tour.
                - 'profit' (float): Item profit.
                - 'weight' (float): Item weight.
                - 'suffix' (float): Remaining tour length from this city to the end.
            - float: Total length of the tour (suffix at position 0).
    """
    n = inst["dim"]
    city_pos = np.empty(n, int)
    for pos, city in enumerate(order):
        city_pos[city] = pos

    suffix = np.empty(len(dists))
    run = 0.0
    for i in reversed(range(len(dists))):
        run += dists[i]
        suffix[i] = run

    recs = []
    for idx, (p, w, c) in enumerate(inst["items"]):
        pos = city_pos[c]
        recs.append({
            "index": idx,
            "city": c,
            "pos": pos,
            "profit": p,
            "weight": w,
            "suffix": float(suffix[pos])
        })

    return recs, float(suffix[0])

# ===============================================================
# EVALUATION
# ===============================================================
def evaluate(inst, order, dists, recs, chosen):
    """
    Evaluates a TTP solution vectorised over cities, computing travel time, profit, and carried weight.

    Args:
        inst (dict): TTP instance dictionary containing:
            - 'capacity' (float): Knapsack capacity.
            - 'v_min' (float): Minimum velocity.
            - 'v_max' (float): Maximum velocity.
        order (np.ndarray): Tour order as a sequence of city indices.
        dists (np.ndarray): Distances between consecutive cities in the tour.
        recs (list[dict]): Preprocessed item records.
        chosen (np.ndarray): Boolean array indicating which items are selected.

    Returns:
        tuple:
            - float: Total travel time (T) for the tour considering carried weight.
            - float: Total profit (P) from chosen items.
            - float: Total weight (W) carried along the tour.
    """
    cap  = inst["capacity"]
    vmin = inst["v_min"]
    vmax = inst["v_max"]

    n = len(order)

    # Build profit/weight per city (vectorised gather)
    chosen_arr = np.asarray(chosen, dtype=bool)

    # Extract fields as arrays
    pos   = np.array([r["pos"]    for r in recs], dtype=int)
    w_arr = np.array([r["weight"] for r in recs], dtype=float)
    p_arr = np.array([r["profit"] for r in recs], dtype=float)

    # Only chosen items contribute
    w_chosen = w_arr[chosen_arr]
    p_chosen = p_arr[chosen_arr]
    pos_ch   = pos[chosen_arr]

    # Accumulate weights & profits into city positions
    city_w = np.zeros(n, float)
    city_p = np.zeros(n, float)
    np.add.at(city_w, pos_ch, w_chosen)
    np.add.at(city_p, pos_ch, p_chosen)

    # Vectorised cumulative carried weight
    W_cum = np.cumsum(city_w)

    # Speed per segment (vectorised clamp)
    alpha = (vmax - vmin) / cap
    v = vmax - alpha * W_cum
    v = np.maximum(v, vmin)

    T = np.sum(dists / v)
    P = float(np.sum(city_p))
    W = float(W_cum[-1])

    return T, P, W

# ===============================================================
# CAPACITY REPAIR
# ===============================================================
def repair_to_capacity(chosen, recs, cap):
    """
    Repairs a selection of items to satisfy the knapsack capacity constraint.

    If the total weight exceeds the capacity, items are randomly removed until 
    the total weight is less than or equal to the capacity. If the total weight 
    is below capacity, one feasible item is randomly added.

    Args:
        chosen (np.ndarray): Boolean array indicating currently selected items.
        recs (list[dict]): Preprocessed item records.
        cap (float): Knapsack capacity.

    Returns:
        np.ndarray: Repaired selection of items satisfying the capacity constraint.
    """
    w = 0.0
    for i, c in enumerate(chosen):
        if c:
            w += recs[i]["weight"]

    # OVERWEIGHT: remove random items until <= cap
    if w > cap:
        # all items currently taken
        idxs = [i for i, c in enumerate(chosen) if c]

        # random removal order
        rng_global.shuffle(idxs)

        for i in idxs:
            if w <= cap:
                break
            chosen[i] = False
            w -= recs[i]["weight"]

    # UNDERWEIGHT: add EXACTLY ONE random feasible item
    elif w < cap:
        # all items NOT taken
        idxs = [i for i, c in enumerate(chosen) if not c]

        rng_global.shuffle(idxs)

        for i in idxs:
            wt = recs[i]["weight"]
            if w + wt <= cap:
                chosen[i] = True
                w += wt
                break  

    return chosen

# ===============================================================
# GREEDY SEEDING MODES
# ===============================================================

def build_seed(inst, recs, total_dist):
    """
    Selects items for the TTP using a randomised greedy heuristic.

    The heuristic chooses a mode at random ("suffix", "profit_adj", or "speed_adj") 
    and scores each item accordingly. Items are then selected in descending score 
    order until a target capacity fraction is reached, followed by a repair step 
    to ensure feasibility.

    Args:
        inst (dict): TTP instance dictionary.
        recs (list[dict]): Preprocessed item records.
        total_dist (float): Total length of the tour.

    Returns:
        list[bool]: Boolean array indicating which items are selected.
    """
    mode = rng_greedyseed.choice(HEUR_MODES)
    gamma = rng_greedyseed.choice(GAMMAS)
    power = rng_greedyseed.choice([0.5, 1.0, 2.0])
    cap_target = rng_greedyseed.choice(CAP_FACTORS) * inst["capacity"]

    if mode == "suffix":
        for r in recs:
            r["score"] = (r["suffix"] / total_dist) ** power

    elif mode == "profit_adj":
        for r in recs:
            adj = r["weight"] * (1.0 + gamma * (r["suffix"] / total_dist))
            r["score"] = r["profit"] / adj

    else:
        alpha = (inst["v_max"] - inst["v_min"]) / inst["capacity"]
        for r in recs:
            dv = alpha * r["weight"]
            v_before = inst["v_max"]
            v_after = max(inst["v_max"] - dv, inst["v_min"])
            dt = r["suffix"] / v_after - r["suffix"] / v_before
            if dt <= 0:
                dt = 1e-6
            r["score"] = r["profit"] / dt

    idxs = sorted(range(len(recs)), key=lambda i: recs[i]["score"], reverse=True)

    chosen = [False] * len(recs)
    W = 0.0
    for i in idxs:
        w = recs[i]["weight"]
        if W + w <= cap_target:
            chosen[i] = True
            W += w

    chosen = repair_to_capacity(chosen, recs, inst["capacity"])
    return chosen

# ===============================================================
# MUTATION / CROSSOVER
# ===============================================================
def mutate(chosen, rate=MUT_RATE):
    """
    Applies a bit-flip mutation to a selection of items.

    Args:
        chosen (np.ndarray): Current selection of items as a boolean array.
        rate (float, optional): Mutation probability per item. Defaults to MUT_RATE.

    Returns:
        list or np.ndarray: Mutated selection of items.
    """
    out = chosen[:]
    for i in range(len(out)):
        if rng_mutation.random() < rate:
            out[i] = not out[i]
    return out

def crossover(a, b):
    """
    Performs single-point crossover between two parent selections.

    Args:
        a (list or np.ndarray): First parent selection.
        b (list or np.ndarray): Second parent selection.

    Returns:
        list: Offspring selection resulting from the crossover.
    """
    n = len(a)
    cut = rng_crossover.randint(0, n - 1)
    return a[:cut] + b[cut:]


# ===============================================================
# NSGA-II
# ===============================================================
def dominates(A, B):
    """
    Checks if solution A dominates solution B. A dominates B if A is no worse in all objectives and strictly better in at least one.

    Args:
        A (tuple): Objective values of solution A as (time, profit, weight).
        B (tuple): Objective values of solution B as (time, profit, weight).

    Returns:
        bool: True if A dominates B, False otherwise.
    """
    t1, p1, _ = A
    t2, p2, _ = B
    return (t1 <= t2 and p1 >= p2) and (t1 < t2 or p1 > p2)

def pareto_rank(objs):
    """
    Computes Pareto fronts for a set of solutions. Solutions are ranked into successive non-dominated fronts.

    Args:
        objs (list): Objective values (time, profit, weight) for each solution.

    Returns:
        list: List of fronts; each front is a list of indices of solutions belonging to that front.
    """
    n = len(objs)
    dom_counts = [0] * n
    dominated_by = [[] for _ in range(n)]
    fronts = [[]]

    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            if dominates(objs[i], objs[j]):
                dominated_by[i].append(j)
            elif dominates(objs[j], objs[i]):
                dom_counts[i] += 1
        if dom_counts[i] == 0:
            fronts[0].append(i)

    k = 0
    while fronts[k]:
        nxt = []
        for i in fronts[k]:
            for j in dominated_by[i]:
                dom_counts[j] -= 1
                if dom_counts[j] == 0:
                    nxt.append(j)
        k += 1
        fronts.append(nxt)

    return fronts[:-1]

def crowding_distance(front_indices, objs):
    """
    Computes crowding distances for a Pareto front.

    Args:
        front_indices (list): Indices of solutions in the front.
        objs (list): Objective values (time, profit, weight) for each solution.

    Returns:
        dict: Mapping from solution index to crowding distance (float). Boundary solutions get infinite distance.
    """
    if not front_indices:
        return {}

    # init distances
    dist = {i: 0.0 for i in front_indices}

    # use only time (index 0, minimise) and profit (index 1, maximise)
    num_obj = 2
    for m in range(num_obj):
        vals = [(i, objs[i][m]) for i in front_indices]

        if m == 0:
            # time: smaller is better -> sort ascending
            vals.sort(key=lambda x: x[1])
        else:
            # profit: larger is better -> sort descending
            vals.sort(key=lambda x: x[1], reverse=True)

        v_min = vals[0][1]
        v_max = vals[-1][1]
        if v_max == v_min:
            # no spread on this dimension
            continue

        # boundary points get infinite crowding
        dist[vals[0][0]]  = float("inf")
        dist[vals[-1][0]] = float("inf")

        # internal points
        for k in range(1, len(vals) - 1):
            i_prev, v_prev = vals[k - 1]
            i_curr, v_curr = vals[k]
            i_next, v_next = vals[k + 1]
            incr = (v_next - v_prev) / (v_max - v_min)
            dist[i_curr] += incr

    return dist

# ===============================================================
# DIVERSITY-AWARE MATING SCORES + TOURNAMENT SELECTION
# ===============================================================
def compute_mating_scores(objs, alpha=0.5, beta=0.3, gamma=0.2):
    """
    Computes a mating probability score for each individual in a population based on 
    multi-objective optimization criteria: Pareto rank, crowding distance, and sparsity.

    Args:
        objs (list): List of objective values (T, P, W) for each individual.
        alpha (float, optional): Weight for Pareto rank (default 0.5).
        beta (float, optional): Weight for crowding distance (default 0.3).
        gamma (float, optional): Weight for sparsity from population centroid (default 0.2).

    Returns:
        list: List of score (float), one per individual.
    """

    n = len(objs)
    if n == 0:
        return []

    # Pareto rank score (front 0 best)
    fronts = pareto_rank(objs)  # list of fronts, each is list of indices
    rank_score = [0.0] * n
    if len(fronts) == 1:
        # everyone same front
        for i in fronts[0]:
            rank_score[i] = 1.0
    else:
        max_front = max(1, len(fronts) - 1)
        for rank, front in enumerate(fronts):
            val = 1.0 - (rank / max_front)   # front 0 -> 1.0, last front -> ~0.0
            for i in front:
                rank_score[i] = val

    # Crowding score over whole pop
    idxs = list(range(n))
    cd = crowding_distance(idxs, objs)
    cd_vec = [cd.get(i, 0.0) for i in range(n)]

    # normalise crowding, keep boundary INF higher than others
    finite_vals = [v for v in cd_vec if math.isfinite(v)]
    if not finite_vals:
        cd_norm = [0.0] * n
    else:
        max_fin = max(finite_vals)
        if max_fin <= 0.0:
            cd_norm = [0.0] * n
        else:
            cd_norm = []
            for v in cd_vec:
                if not math.isfinite(v):
                    cd_norm.append(1.1)  # slightly above max finite
                else:
                    cd_norm.append(v / max_fin)

    # Sparsity: distance from centroid in (T,P)
    times = np.array([o[0] for o in objs], dtype=float)
    profits = np.array([o[1] for o in objs], dtype=float)

    t_bar = float(times.mean())
    p_bar = float(profits.mean())

    div_radius = np.sqrt((times - t_bar) ** 2 + (profits - p_bar) ** 2)
    max_r = float(div_radius.max()) if n > 0 else 0.0
    if max_r <= 0.0:
        div_norm = [0.0] * n
    else:
        div_norm = [float(r / max_r) for r in div_radius]

    # Combine into final score
    scores = []
    for i in range(n):
        s = (alpha * rank_score[i] +
             beta  * cd_norm[i] +
             gamma * div_norm[i])
        scores.append(s)

    return scores

def tournament_select(pop, scores, k=TOURN_K):
    """
    Performs tournament selection on a population using precomputed scores.

    Args:
        pop (list): Population of individuals.
        scores (list[float]): Mating or fitness scores corresponding to each individual.
        k (int, optional): Tournament size (default TOURN_K).

    Returns:
        int: Index of the selected individual.
    """
    n = len(pop)
    if n == 0:
        raise ValueError("tournament_select called with empty population")
    k = min(k, n)

    cand = rng_selection.sample(range(n), k)
    best_idx = max(cand, key=lambda i: scores[i])
    return best_idx

def vns_hill_climb_knapsack(sol, inst, order, dists, recs, max_steps: int):
    """
    Performs a Variable Neighborhood Search (VNS) hill climbing on a knapsack solution 
    for the Traveling Thief Problem.

    The algorithm explores multiple neighborhood sizes, adapts if stuck in a local basin, 
    and only accepts solutions that improve profit or maintain profit while reducing travel time. 
    Each candidate solution is repaired to satisfy the knapsack capacity.

    Args:
        sol (list[bool]): Initial selection of items.
        inst (dict): TTP instance dictionary containing at least 'capacity'.
        order (list[int]): Tour order as a sequence of city indices.
        dists (np.ndarray): Distances between consecutive cities in the tour.
        recs (list[dict]): Preprocessed item records.
        max_steps (int): Maximum number of hill climbing iterations.

    Returns:
        tuple:
            - list[bool]: Improved item selection after hill climbing.
            - tuple(float, float, float): Corresponding (T, P, W) values of the best solution.
    """

    best = sol[:]
    best_T, best_P, best_W = evaluate(inst, order, dists, recs, best)

    n_items = len(best)

    # initial neighbourhood identical to old version
    k_min, k_max = 1, 4

    stuck_counter = 0

    for _ in range(max_steps):

        # widen neighbourhood as % of HC budget
        if stuck_counter > 0.10 * max_steps:
            k_min, k_max = 1, 6    # mild escape

        if stuck_counter > 0.30 * max_steps:
            k_min, k_max = 1, 8    # stronger escape

            k_min, k_max = 1, 8

        # pick k
        k = rng_hillclimb.randint(k_min, k_max)
        idxs = rng_hillclimb.sample(range(n_items), k)

        # generate neighbour
        cand = best[:]
        for j in idxs:
            cand[j] = not cand[j]

        cand = repair_to_capacity(cand, recs, inst["capacity"])
        T2, P2, W2 = evaluate(inst, order, dists, recs, cand)

        # accept if non-dominated wrt time/profit
        if P2 > best_P or (P2 == best_P and T2 < best_T):
            best = cand[:]
            best_T, best_P, best_W = T2, P2, W2
            stuck_counter = 0        # reset
        else:
            stuck_counter += 1       # track stagnation

    return best, (best_T, best_P, best_W)

def select_top_from_archive(archive_objs, archive_pop, target=MAX_ARCHIVE):
    """
    Performs ε-dominance thinning to select a fixed number of solutions from an archive.

    This method guarantees exact target solutions are selected, preserving both 
    diversity and quality in bi-objective (time, profit) space. 

    Args:
        archive_objs (list[tuple]): List of objective tuples (T, P, W) for archived solutions.
        archive_pop (list): Corresponding population of solutions in the archive.
        target (int, optional): Number of solutions to retain. Defaults to MAX_ARCHIVE.

    Returns:
        tuple:
            - list: Selected population of solutions.
            - list: Corresponding objective tuples of the selected solutions.
    """
    objs = archive_objs[:]
    pop  = archive_pop[:]

    T = np.array([o[0] for o in objs])
    P = np.array([o[1] for o in objs])
    Tmin, Tmax = T.min(), T.max()
    Pmin, Pmax = P.min(), P.max()

    norm = np.column_stack(((T-Tmin)/(Tmax-Tmin+1e-12),
                            (P-Pmin)/(Pmax-Pmin+1e-12)))

    eps = 0.30
    survivors = set()

    while eps > 1e-4 and len(survivors) < target:
        buckets = {}
        for i,(x,y) in enumerate(norm):
            key = (int(x/eps), int(y/eps))
            if key not in buckets:
                buckets[key] = i
        survivors = set(buckets.values())
        if len(survivors) < target:
            eps *= 0.75

    survivors = list(survivors)

    if len(survivors) > target:
        pts = norm[survivors]
        dmat = np.sqrt(((pts[:,None,:] - pts[None,:,:])**2).sum(2)) + np.eye(len(pts))*1e9
        crowd = dmat.min(1)
        survivors = [survivors[i] for i in np.argsort(-crowd)[:target]]

    if len(survivors) < target:
        remaining = list(set(range(len(objs))) - set(survivors))
        pts_keep = norm[survivors]
        fill = []
        for i in remaining:
            d = np.sqrt(((pts_keep - norm[i])**2).sum(1)).min()
            fill.append((d,i))
        fill = sorted(fill, reverse=True)
        survivors += [idx for (_,idx) in fill[:target-len(survivors)]]

    survivors = survivors[:target]
    return [pop[i] for i in survivors], [objs[i] for i in survivors]

# ===============================================================
# EVOLUTION LOOP
# ===============================================================
def run_evo(inst, order, dists, recs, total_dist, cand,
            gens=GENERATIONS, pop_size=POP_SIZE):
    """
    Runs a multi-objective evolutionary algorithm for the 
    TTP, combining knapsack optimization, TSP local search, and NSGA-II selection.

    Args:
        inst (dict): TTP instance containing 'coords', 'items', 'capacity', 
                    'v_min', 'v_max', and 'dim' (number of cities).
        order (list or ndarray): Initial TSP tour order.
        dists (ndarray): Precomputed distances along the tour.
        recs (list): Item records with positional info and suffix sums.
        total_dist (float): Total distance of the initial tour.
        cand (list of lists): K-nearest neighbour candidate lists per city.
        gens (int, optional): Number of evolutionary generations (default: GENERATIONS).
        pop_size (int, optional): Population size (default: POP_SIZE).

    Returns:
        tuple: 
            - list of list of bool: Final population of knapsack solutions.
            - list of tuple: Corresponding objectives (time, profit, weight) for each individual.
    """

    global archive_pop, archive_objs, bestP, bestP_sol, bestT, bestT_sol

    # Initialise tours exactly once for the starting population
    tours = [order.tolist() if isinstance(order, np.ndarray) else order[:]
            for _ in range(pop_size)]
    
    # Initial population
    pop = [build_seed(inst, recs, total_dist)
           for _ in range(pop_size)]
    objs = [evaluate(inst, order, dists, recs, c) for c in pop]

    for g in range(gens):
        # Update global best profit & best time from current population
        for sol, (T, P, W) in zip(pop, objs):
            if P > bestP:
                bestP = P
                bestP_sol = sol[:]
            if bestT is None or T < bestT:
                bestT = T
                bestT_sol = sol[:]

        print(f"[GEN {g:4d}] min_time={min(o[0] for o in objs):.1f}  max_profit={max(o[1] for o in objs):.0f}")


        # IDENTIFY ELITES FROM CURRENT POPULATION
        num_tail_elites = TAIL_ELITES
        num_mid_elites  = MID_ELITES
        tail_pool_size  = min(TAIL_POP, len(objs))

        elite_indices = []

        if num_mid_elites > 0:
            rand_mid = rng_selection.sample(range(len(objs)), num_mid_elites)
            for i in rand_mid:
                if i not in elite_indices:
                    elite_indices.append(i)

        # RANDOM TAIL ELITES
        idx_sorted = sorted(
            range(len(objs)),
            key=lambda i: (objs[i][0], -objs[i][1])
        )

        tail_pool = idx_sorted[:tail_pool_size]

        if num_tail_elites > 0:
            chosen_tail = rng_selection.sample(tail_pool, num_tail_elites)
            for i in chosen_tail:
                if i not in elite_indices:
                    elite_indices.append(i)

        elite_solutions = [pop[i][:] for i in elite_indices]
        elite_objs      = [objs[i]   for i in elite_indices]

        # ADD SPARSE ARCHIVE ELITES
        if len(archive_pop) > 0:
            # compute sparsity = nearest-neighbour distance in (T,P)
            sparsity = []
            for i in range(len(archive_pop)):
                obj_i = archive_objs[i]
                Ti = obj_i[0]
                Pi = obj_i[1]

                best_d = float("inf")
                for j in range(len(archive_pop)):
                    if i == j:
                        continue
                    obj_j = archive_objs[j]
                    Tj = obj_j[0]
                    Pj = obj_j[1]

                    d = (Ti - Tj)**2 + (Pi - Pj)**2
                    if d < best_d:
                        best_d = d

                sparsity.append(best_d)


            # sort by sparsity descending (most isolated first)
            idx_sorted = sorted(
                range(len(archive_pop)),
                key=lambda i: sparsity[i],
                reverse=True
            )

            # top SPARSE_TAIL_POP sparsest form the candidate pool
            pool_size = min(SPARSE_POP, len(archive_pop))
            sparse_pool = idx_sorted[:pool_size]

            # random sample SPARSE_ELITES from this pool
            k = min(SPARSE_ELITES, pool_size)
            chosen_sparse = rng_selection.sample(sparse_pool, k)

            # append selected sparse elites
            for idx in chosen_sparse:
                elite_solutions.append(archive_pop[idx][:])
                elite_objs.append(archive_objs[idx])
                elite_indices.append(-1)   # dummy index to maintain structure


        # PREPARE/RESTORE TOUR CACHE FOR ELITE PROCESSING
        tour_cache = getattr(run_evo, "_tour_cache", {})

        # safety cap
        if len(tour_cache) > MAX_TOUR_CACHE:
            tour_cache.clear()

        setattr(run_evo, "_tour_cache", tour_cache)



        # RANDOMLY DO KNAPSACK OR TSP IMPROVEMENT
        for e, sol in enumerate(elite_solutions):

            do_tsp = rng_global.random() < 0.5

            if not do_tsp:
                # KNAPSACK ONLY
                sol_refined, (best_T, best_P, best_W) = vns_hill_climb_knapsack(
                    sol,
                    inst,
                    order,
                    dists,
                    recs,
                    max_steps=HC_STEPS,
                )

                elite_solutions[e] = sol_refined[:]
                elite_objs[e]      = (best_T, best_P, best_W)

                idx = elite_indices[e]
                if idx == -1:
                    pop.append(sol_refined[:]); objs.append((best_T, best_P, best_W))
                else:
                    pop[idx] = sol_refined[:]; objs[idx] = (best_T, best_P, best_W)

            else:
                #  TSP ONLY
                key = tuple(sol)
                tour = tour_cache.get(
                    key,
                    order.tolist() if isinstance(order, np.ndarray) else order[:]
                )[:]

                # BASE LOCAL SEARCH (LS CYCLE)
                # 2-opt -> 3-opt -> 2-opt cleanup
                # Pass 1: strong 2-opt
                tour, _ = two_opt_cy(
                    tour, cand, inst["coords"],
                    max_iter=ELITE_2OPT_STEPS,
                    verbose=False,
                    seed=rng_global.randint(0, 2**31 - 1)
                )
                # Pass 2: strong 3-opt
                tour, _ = three_opt_cy(
                    tour, cand, inst["coords"],
                    max_iter=ELITE_3OPT_STEPS // 2,
                    verbose=False,
                    seed=rng_global.randint(0, 2**31 - 1)
                )
                # Pass 3: cleanup 2-opt — required after 3-opt
                tour, _ = two_opt_cy(
                    tour, cand, inst["coords"],
                    max_iter=ELITE_2OPT_STEPS // 2,
                    verbose=False,
                    seed=rng_global.randint(0, 2**31 - 1)
                )

                # Evaluate basin after base LS
                _, base_len = get_tour_dists(tour, inst["coords"])

                # ILS LOOP (corrected with real kicks)
                # double-bridge -> 2-opt -> 3-opt -> 2-opt cleanup
                for r in range(ELITE_ILS_PASSES):

                    # Choose kick strength by phase 
                    if r < ELITE_ILS_PASSES // 3:
                        cuts = ILS_KICK_CUTS_BASE
                        max2 = ELITE_2OPT_STEPS
                        max3 = ELITE_3OPT_STEPS
                    elif r < 2 * ELITE_ILS_PASSES // 3:
                        cuts = ILS_KICK_CUTS_MID
                        max2 = ELITE_2OPT_STEPS // 2
                        max3 = ELITE_3OPT_STEPS // 2
                    else:
                        cuts = ILS_KICK_CUTS_LATE
                        max2 = ELITE_2OPT_STEPS // 3
                        max3 = ELITE_3OPT_STEPS // 3

                    # Kick 
                    kicked = double_bridge_move(tour, cuts=cuts)
                    kicked = ensure_valid_tour(kicked, len(tour))

                    # LS Pass 1: phased 2-opt
                    kicked, _ = two_opt_cy(
                        kicked, cand, inst["coords"],
                        max_iter=max2,
                        verbose=False,
                        seed=rng_tourkick.randint(0, 2**31 - 1)
                    )

                    # LS Pass 2: phased 3-opt
                    kicked, _ = three_opt_cy(
                        kicked, cand, inst["coords"],
                        max_iter=max3,
                        verbose=False,
                        seed=rng_tourkick.randint(0, 2**31 - 1)
                    )

                    # LS Pass 3: cleanup 2-opt
                    kicked, _ = two_opt_cy(
                        kicked, cand, inst["coords"],
                        max_iter=max2 // 2,
                        verbose=False,
                        seed=rng_tourkick.randint(0, 2**31 - 1)
                    )

                    # Evaluate 
                    _, k_len = get_tour_dists(kicked, inst["coords"])

                    if k_len < base_len:
                        tour = kicked[:]
                        base_len = k_len

                # SAVE UPDATED TOUR & OBJECTIVES
                tour_cache[key] = tour[:]

                d2, _ = get_tour_dists(tour, inst["coords"])
                T2, P2, W2 = evaluate(inst, tour, d2, recs, sol)

                elite_solutions[e] = sol[:]
                elite_objs[e]      = (T2, P2, W2)

                idx = elite_indices[e]
                pop[idx]  = sol[:]
                objs[idx] = (T2, P2, W2)
                tours[idx] = tour[:]


        # Inject fresh greedy individuals into population
        fresh = []

        for _ in range(NUM_GREEDIES):
            seed = build_seed(inst, recs, total_dist)
            fresh.append(seed)

        fresh_objs = [evaluate(inst, order, dists, recs, s) for s in fresh]

        # add to population pre-selection
        pop  += fresh
        objs += fresh_objs

        while len(tours) < len(pop):
            base = order.tolist() if isinstance(order, np.ndarray) else order[:]
            tours.append(base[:])

        # DIVERSITY-AWARE OFFSPRING GENERATION
        # parents chosen by combined (rank + crowding + sparsity) score
        mating_scores = compute_mating_scores(objs)

        # OFFSPRING — inherit parent tours correctly
        offspring = []
        offspring_tours = []

        while len(offspring) < pop_size:

            # Select two parents
            idx_a = tournament_select(pop, mating_scores, k=TOURN_K)
            idx_b = tournament_select(pop, mating_scores, k=TOURN_K)

            if idx_b == idx_a and len(pop) > 1:
                idx_b = rng_selection.randrange(len(pop))

            parent_a = pop[idx_a]
            parent_b = pop[idx_b]

            # Knapsack recombination
            child = crossover(parent_a, parent_b)
            child = mutate(child)
            child = repair_to_capacity(child, recs, inst["capacity"])

            dA,_ = get_tour_dists(tours[idx_a], inst["coords"])
            dB,_ = get_tour_dists(tours[idx_b], inst["coords"])
            Ta,_,_ = evaluate(inst, tours[idx_a], dA, recs, child)
            Tb,_,_ = evaluate(inst, tours[idx_b], dB, recs, child)

            # Tour inheritance, choose the parent whose tour gives the lower time
            if Ta < Tb:
                chosen_tour = tours[idx_a]
            else:
                chosen_tour = tours[idx_b]



            offspring.append(child)
            offspring_tours.append(chosen_tour[:])

        # child objs evaluated AFTER tours assigned — correct behaviour
        off_objs = [evaluate(inst, t, dists, recs, c)
                    for t, c in zip(offspring_tours, offspring)]

        # NSGA-II ENVIRONMENTAL SELECTION ON POP ∪ OFFSPRING
        combined = pop + offspring
        comb_objs = objs + off_objs

        fronts = pareto_rank(comb_objs)

        new_pop = []
        new_objs = []

        for front in fronts:
            if len(new_pop) + len(front) <= pop_size:
                for idx in front:
                    new_pop.append(combined[idx])
                    new_objs.append(comb_objs[idx])
            else:
                need = pop_size - len(new_pop)
                picks = rng_selection.sample(front, need)
                for idx in picks:
                    new_pop.append(combined[idx])
                    new_objs.append(comb_objs[idx])
                break

        # RE-INSERT ELITES INTO POPULATION
        # replace worst individuals if missing
        def worst_index(objs_list):
            return max(range(len(objs_list)),
                       key=lambda i: (objs_list[i][0], -objs_list[i][1]))

        for e_sol, e_obj in zip(elite_solutions, elite_objs):
            # Membership check by objective tuple
            if e_obj not in new_objs:
                wi = worst_index(new_objs)
                new_pop[wi] = e_sol[:]
                new_objs[wi] = e_obj

        # SYNCHRONISE elite_tours
        pop, objs = new_pop, new_objs

        # PRUNE _tour_cache TO ONLY (elite + ND archive) KEYS
        tour_cache = getattr(run_evo, "_tour_cache", {})

        # Build the only allowed keys
        allowed_keys = set()

        # All elites this generation
        for sol in elite_solutions:
            allowed_keys.add(tuple(sol))

        # All ND archive individuals
        for sol in archive_pop:
            allowed_keys.add(tuple(sol))

        # Rebuild the cache
        pruned_cache = {}
        for key in allowed_keys:
            if key in tour_cache:
                pruned_cache[key] = tour_cache[key]

        # Replace with pruned version
        setattr(run_evo, "_tour_cache", pruned_cache)


        # Rebuild tours to match the new population order
        tours = []
        for sol in pop:
            key = tuple(sol)
            if key in run_evo._tour_cache:
                tours.append(run_evo._tour_cache[key][:])
            else:
                tours.append(order.tolist() if isinstance(order, np.ndarray) else order[:])

        # replace worst-for-that-objective if global elite is missing

        # Worst by profit (lowest profit)
        def worst_by_profit(objs_list):
            return min(range(len(objs_list)), key=lambda i: objs_list[i][1])

        # Worst by time (largest time)
        def worst_by_time(objs_list):
            return max(range(len(objs_list)), key=lambda i: objs_list[i][0])

        # profit-elite
        if bestP_sol is not None:
            Tp, Pp, Wp = evaluate(inst, order, dists, recs, bestP_sol)
            eliteP_obj = (Tp, Pp, Wp)
            if eliteP_obj not in objs:
                wi = worst_by_profit(objs)
                pop[wi] = bestP_sol[:]
                objs[wi] = eliteP_obj

        # time-elite
        if bestT_sol is not None:
            Tt, Pt, Wt = evaluate(inst, order, dists, recs, bestT_sol)
            eliteT_obj = (Tt, Pt, Wt)
            if eliteT_obj not in objs:
                wi = worst_by_time(objs)
                pop[wi] = bestT_sol[:]
                objs[wi] = eliteT_obj


        # UPDATE PERSISTENT ND ARCHIVE

        objs_with_gen = [(T,P,W,g) for (T,P,W) in objs]
        combo_objs = archive_objs + objs_with_gen

        # find global ND set over all time
        fronts = pareto_rank([(t,p,w) for (t,p,w,_) in combo_objs])
        nd = fronts[0]  # keep indices of ND only

        # Build new archive lists
        newA_pop  = []
        newA_objs = []

        for i in nd:
            if i < len(archive_objs): # came from previous archive
                newA_pop.append(archive_pop[i])
                newA_objs.append(archive_objs[i])  
            else: # came from this generation
                j = i - len(archive_objs)
                newA_pop.append(pop[j])
                newA_objs.append(objs_with_gen[j])

        # remove duplicates based on (rounded T,P)
        uniq = {}
        for sol, obj in zip(newA_pop,newA_objs):
            T,P,W,_ = obj
            key = (round(T,3), round(P,1))
            if key not in uniq or W < uniq[key][1][2]:
                uniq[key] = (sol, obj)

        archive_pop  = [v[0] for v in uniq.values()]
        archive_objs = [v[1] for v in uniq.values()]  

    return pop, objs

# ===============================================================
# HYPERVOLUME
# ===============================================================

def compute_hypervolume(objs, ideal, nadir):
    """
    Calculates the hypervolume of a set of solutions. The hypervolume is computed with respect to a given ideal and nadir point,
    clipping contributions outside the nadir. 

    Args:
        objs (list of tuple): List of solutions as (T, P, W) or (T, P, W, GEN).
        ideal (tuple): Ideal point (T_min, -P_max) for hypervolume calculation.
        nadir (tuple): Nadir point (T_max, -P_min) for hypervolume calculation.

    Returns:
        float: Hypervolume value representing the dominated area in the (time, -profit) plane.
    """

    pts = []
    for x in objs:
        # Accept either length-3 or length-4 tuples
        if len(x) == 3:
            T,P,_ = x
        else:
            T,P,_,_ = x  # GEN ignored for HV
        pts.append((T, -P))

    # clip: only keep pts with T <= NadirT and profit >= -IdealP
    clipped = [(T,p) for (T,p) in pts if (T <= nadir[0] and p >= ideal[1])]

    if not clipped:
        return 0.0

    clipped.sort(key=lambda x: x[0])  # sort by time asc
    hv = 0.0
    prev_t = ideal[0]

    for (t,p) in clipped:
        width  = max(0, t - prev_t)
        height = max(0, nadir[1] - p)
        hv += width * height
        prev_t = t

    width  = max(0, nadir[0] - prev_t)
    height = max(0, nadir[1] - clipped[-1][1])
    hv += width * height

    return hv

In [ ]:
# ===============================================================
# Load Instance and Build NN Tour
# ===============================================================
print("[INFO] Loading TTP instance…")
inst = parse_ttp(INSTANCE_PATH)

print("[INFO] Building NN tour…")
order = nn_tour(inst["coords"])
dists, _ = get_tour_dists(order, inst["coords"])
print(f"[TSP] NN length = {dists.sum():.0f}")

In [ ]:
# ===============================================================
# Run LKH Style TSP
# ===============================================================

print("[INFO] Running LKH-style TSP search (C++ optimisation)…")
order_list, tsp_len, cand = lkh_style_tsp(inst, order)


# convert to numpy array because the rest of your TTP uses numpy for 'order'
order = np.array(order_list, dtype=int)
dists, _ = get_tour_dists(order, inst["coords"])
print(f"[TSP] LKH tour length = {dists.sum():.0f}")



In [ ]:
# ===============================================================
# Run EVO Loop
# ===============================================================

print("[INFO] Preprocessing items…")
recs, total_dist = prep_items(inst, order, dists)

pop, objs = run_evo(inst, order, dists, recs, total_dist, cand,
                    gens=GENERATIONS, pop_size=POP_SIZE)



In [ ]:
# ===============================================================
# Results and Submission
# ===============================================================

# SELECT FINAL 20
final_pop, final_objs = select_top_from_archive(archive_objs, archive_pop, target=MAX_ARCHIVE)

print(f"\n=== FINAL SUBMISSION SET ({MAX_ARCHIVE}) ===")
for (T,P,W,G) in sorted(final_objs, key=lambda x: x[0]):
    print(f"time={T:.1f}  profit={P:.0f}  weight={W:.0f}  GEN={G}")

print("\n=== ARCHIVE SUMMARY =======================")
print(f"ND archive size: {len(archive_objs)}")
for (T,P,W,G) in sorted(archive_objs, key=lambda x:x[0]):
    print(f"time={T:.1f}  profit={P:.0f}  weight={W:.0f}  GEN={G}")

# HYPERVOLUME (using problem-specific IDEAL / NADIR / P range)
# Time bounds from IDEAL / NADIR
T_min, T_max = IDEAL[0], NADIR[0]
range_t = T_max - T_min

# Profit bounds from CSV (typically P_min = 0, P_max = -IDEAL[1])
range_p = P_max - P_min

# HV on archive (in (T, -P) space, clipped by IDEAL/NADIR)
hv_raw_nd = compute_hypervolume(final_objs, IDEAL, NADIR)
hv_norm_nd = hv_raw_nd / (range_t * range_p)

print("\n=== SUBMISSION HYPERVOLUME (ARCHIVE) ===")
print(f"Raw HV   : {hv_raw_nd:.6e}")
print(f"Norm HV  : {hv_norm_nd:.6f}")


In [ ]:
import matplotlib.pyplot as plt

pts = np.array([(t,p,g) for (t,p,w,g) in final_objs])
pts = pts[pts[:,0].argsort()]  # sort by time

times   = pts[:,0]
profits = pts[:,1]

plt.figure(figsize=(10,7))

#  Dominated area shading for the displayed 20 only
for i in range(len(times)):
    xs = [times[i], T_max, T_max, times[i]]
    ys = [P_min, P_min, profits[i], profits[i]]
    plt.fill(xs, ys, color="lightblue", alpha=0.15)

# Hypervolume boundary (step front)
hv_x = [T_min]
hv_y = [P_min]

for t,p in zip(times, profits):
    hv_x += [t, t]
    hv_y += [hv_y[-1], p]

hv_x.append(T_max)
hv_y.append(hv_y[-1])

plt.plot(hv_x, hv_y, color="pink", linewidth=2.3,
         label=f"HV step boundary ({MAX_ARCHIVE}-set)")

# Pareto front scatter
plt.plot(times, profits, "-o", color="blue", markersize=6,
         markeredgecolor="black", label="Selected 100")

# Best markers
best_t = np.argmin(times)
best_p = np.argmax(profits)

plt.scatter(times[best_t], profits[best_t], s=140,
            color="red", edgecolor="black", label="Fastest")
plt.scatter(times[best_p], profits[best_p], s=140,
            color="green", edgecolor="black", label="Most profitable")

plt.xlim(T_min, T_max)
plt.ylim(P_min, P_max)
plt.grid(alpha=0.3)
plt.xlabel("Time (lower better)")
plt.ylabel("Profit (higher better)")
plt.title(f"Top-{MAX_ARCHIVE} Pareto Submission Set {File} — Norm HV  : {hv_norm_nd:.6f}")
plt.legend(loc="lower right", bbox_to_anchor=(1.0, 0.0))
plt.show()


In [ ]:
# ===============================================================
# REBUILD FINAL TOURS FROM ARCHIVE, NOT RANDOM REPLACEMENTS
# ===============================================================

tour_cache = getattr(run_evo, "_tour_cache", {})

final_tours = []

# Build a map from archive solutions -> their tours
archive_map = {}
for sol in archive_pop:
    key = tuple(sol)
    if key in tour_cache:
        archive_map[key] = tour_cache[key][:]
    else:
        # Should basically never happen unless TSP never ran for that solution
        archive_map[key] = order.tolist() if isinstance(order, np.ndarray) else order[:]

# Now attach the correct tour to the selected final_pop
for sol in final_pop:
    key = tuple(sol)
    final_tours.append(archive_map[key][:])

assert len(final_tours) == len(final_pop), \
    f"tour count mismatch: {len(final_tours)} vs {len(final_pop)}"

print(f"[INFO] Reconstructed {len(final_tours)} tours from archive")



In [ ]:
import os

TEAM = "Group_5"
INSTANCE = File.replace(".txt", "")   # e.g. "a280-n2790"

out_dir = os.path.join("submissions", TEAM)
os.makedirs(out_dir, exist_ok=True)

path_x = os.path.join(out_dir, f"{TEAM}_{INSTANCE}.x")
path_f = os.path.join(out_dir, f"{TEAM}_{INSTANCE}.f")


# WRITE .f  (time profit)
with open(path_f, "w") as f:
    for (T, P, W, G) in final_objs:
        f.write(f"{T:.6f} {P:.6f}\n")

print("[EXPORT] wrote:", path_f)

# WRITE .x  (tour + knapsack)
with open(path_x, "w") as f:
    for tour, ks in zip(final_tours, final_pop):

        # tour line
        f.write(" ".join(str(x) for x in tour) + "\n")

        # knapsack line (0/1 only)
        f.write(" ".join("1" if b else "0" for b in ks) + "\n")

        # mandatory blank line
        f.write("\n")

print("[EXPORT] wrote:", path_x)
